# Concept Assembly Zones: From Snapshots to Process

*A companion to the SAE tutorial — same model, same question, different lens.*

This notebook is a companion to **Mechanistic_Interpretability_LLM_Tutorial.ipynb**. That tutorial follows the standard mechanistic interpretability workflow:

1. Pick a layer → decompose activations with an SAE → find sparse features → steer with them → measure dark matter.

This tutorial asks a different question. Instead of decomposing *what* the model represents at one layer, we track *how* representations form across all layers.

**The SAE approach**: Pick layer 8 of GPT-2. What features are active here?
**The CAZ approach**: Track a concept across all 12 layers. Where does it form? How wide is the assembly zone? Is there one peak or several?

### A note on effort

The SAE tutorial is built on top of significant prior investment:

- **TransformerLens** — years of engineering to wrap model internals with clean hook points
- **SAELens** — a library for loading, training, and querying Sparse Autoencoders
- **Pre-trained SAE dictionaries** — e.g. the GPT-2 Small SAE used in that tutorial has 24,576 features and was trained on millions of tokens of GPU compute by researchers at EleutherAI and independently
- **Gemma Scope** — a Google DeepMind project training SAEs across every layer of Gemma 2B, at multiple dictionary widths
- **Neuronpedia** — a full platform with a database of pre-computed feature activations, LLM-generated labels, and interactive dashboards

All of that work is real and valuable. But it means the SAE tutorial is also fundamentally: *download things other people built, plug them together, and trust the output.* The features are discovered by a dictionary someone else trained. The labels are generated by a separate LLM. The dashboards live on a third-party platform.

**Not every interpretability problem requires millions of tokens on GPU compute.**

This tutorial uses:
- A standard HuggingFace model load (same as any NLP project)
- ~100 contrastive text pairs per concept (generated with any LLM)
- Fisher-normalized centroid distance, PCA, and first differences — all re-derivable from a linear algebra textbook
- One forward hook: project out a direction, measure what changes

Every number traces directly back to its definition. There is no pre-trained dictionary to trust, no platform to query, no labels generated by a separate model. The insights come from observing what the model does with your data — not from decomposing it against someone else's feature basis.

### The four-act structure (same as the SAE tutorial)

| Act | SAE tutorial | This companion |
|-----|-------------|----------------|
| **1 · Black box** | Model is huge, neurons are polysemantic | Model is huge, concepts assemble across depth — no single "best layer" |
| **2 · Decomposition** | SAE encoder → sparse features at one layer | Separation / coherence / velocity across ALL layers |
| **3 · Causality** | Steer by adding a feature direction | Ablate at the CAZ peak vs. off-peak — where does removal hurt most? |
| **4 · Dark matter** | SAE reconstruction error at one hook point | Multi-modal assembly — what the single-peak assumption misses |

### What you need

- **No TransformerLens, no SAELens, no pre-trained dictionaries.** Raw HuggingFace `transformers` and `rosetta_tools`.
- Same model: **GPT-2 Small** (12 layers, 768 dims).
- Contrastive pairs fetched directly from GitHub — no local data setup.

---

### References

- Henry (2026), "The Concept Assembly Zone: A Layer-Wise Framework for Tracking Concept Formation in Transformers" — [paper](https://github.com/jamesrahenry/Concept_Assembly_Zone)
- Henry (2026), "Concept Ordering and Scaling" — [paper](https://github.com/jamesrahenry/Rosetta_Program)
- `rosetta_tools` — [code](https://github.com/jamesrahenry/Rosetta_Tools)
- SAE companion tutorial — `Mechanistic_Interpretability_LLM_Tutorial.ipynb` in this directory

## 0.0 Install rosetta_tools

Run this cell once before the imports below. Using `%pip` installs directly into the active kernel — no restart needed.

In [ ]:
# ============================================================
# 0.0 Install rosetta_tools from GitHub
# ============================================================
# %pip installs into the active kernel — no restart required.
# Works in Jupyter, JupyterLab, Colab, and VS Code notebooks.

%pip install -q git+https://github.com/jamesrahenry/Rosetta_Tools.git

print("rosetta_tools installed.")

## 0.1 Imports and device

The SAE tutorial needs TransformerLens + SAELens. This tutorial uses:

- **HuggingFace `transformers`** — to load the model and get hidden states at every layer.
- **`rosetta_tools`** — CAZ metric computation (separation, coherence, velocity) and ablation.
- **Standard scientific Python** — numpy, matplotlib, scipy.

In [ ]:
import json
import random

import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoModel, AutoTokenizer

from rosetta_tools.extraction import extract_layer_activations, extract_contrastive_activations
from rosetta_tools.caz import (
    compute_separation,
    compute_coherence,
    compute_velocity,
    compute_layer_metrics,
    find_caz_regions,
)
from rosetta_tools.ablation import (
    DirectionalAblator,
    get_transformer_layers,
    compute_dominant_direction,
    compute_baseline_logits,
    kl_divergence_from_logits,
)
from rosetta_tools.gpu_utils import get_device, get_dtype

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = get_device()
dtype = get_dtype(device)
print("Device:", device, "| Dtype:", dtype)

# 1) The Problem: Why "Pick the Best Layer" Isn't Enough

The SAE tutorial explains the black box problem in terms of **superposition** — too many concepts packed into too few neurons. SAEs address this by learning a sparse, overcomplete dictionary at a single layer.

But there's a second problem that SAEs don't address: **which layer do you pick?**

The standard MI workflow is:

1. Pick a layer (often chosen by probing accuracy or convention).
2. Extract features / directions / probes at that layer.
3. Analyze.

This captures a **snapshot** — the state of the concept at one depth. But transformers are iterative. Each layer reads from and writes to a shared residual stream. A concept that's visible at layer 8 was shaped by layers 0 through 7 before it.

The SAE tutorial uses **layer 8** of GPT-2 Small (67% depth). That's a reasonable choice — it's where many concepts are near peak visibility. But:

- What about concepts that peak earlier? (Credibility peaks around 40% depth across architectures.)
- What about concepts with **multiple peaks**? (We'll see this shortly.)
- What about subtle assembly events that fall below detection thresholds?

The CAZ framework tracks the full assembly process across every layer.

## 1.1 Load the same model

Same model as the SAE tutorial: GPT-2 Small. But we load it via plain HuggingFace `AutoModel` — no TransformerLens wrapper needed. The CAZ framework only requires `output_hidden_states=True`, which every HuggingFace model supports.

In [ ]:
# ============================================================
# 1.1 Load GPT-2 Small (same model as the SAE tutorial)
# ============================================================

MODEL_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=dtype).to(device)
model.eval()

n_layers = model.config.n_layer
n_heads = model.config.n_head
d_model = model.config.n_embd

print(f"Model: {MODEL_NAME}")
print(f"Layers: {n_layers}")
print(f"Heads:  {n_heads}")
print(f"d_model: {d_model}")
print(f"Vocab: {model.config.vocab_size}")
print(f"\nThe SAE tutorial hooks layer 8 ({8/n_layers*100:.0f}% depth).")
print(f"We will track all {n_layers} layers.")

## 1.2 Contrastive data: what CAZ needs that SAEs don't

The SAE tutorial feeds arbitrary prompts through the model and asks "what features fire?" It doesn't need labeled data — the SAE dictionary was pre-trained on billions of tokens.

The CAZ framework needs **contrastive pairs**: text that expresses a concept vs. text that doesn't, matched for topic and style. This is because CAZ measures how well the model's internal geometry **separates** the two classes at each layer.

We'll use consensus pairs from the Rosetta Program dataset — generated by multiple LLMs and filtered for agreement.

In [ ]:
# ============================================================
# 1.2 Load contrastive data for multiple concepts
# ============================================================
# Datasets are fetched directly from GitHub — no local Rosetta_Program
# checkout needed. Same pattern as the SAE tutorial fetching from SAELens/HuggingFace.

import urllib.request

GITHUB_RAW = (
    "https://raw.githubusercontent.com/jamesrahenry/Rosetta_Program/main"
    "/datasets/consensus_pairs"
)

CONCEPTS = ["negation", "sentiment", "credibility", "moral_valence",
            "causation", "certainty", "temporal_order"]

N_PAIRS = 48  # 48 positive + 48 negative = 96 texts per concept

concept_data = {}
for concept in CONCEPTS:
    url = f"{GITHUB_RAW}/{concept}_consensus_pairs.jsonl"
    pos_texts, neg_texts = [], []

    with urllib.request.urlopen(url) as resp:
        for line in resp:
            rec = json.loads(line)
            if rec["label"] == 1 and len(pos_texts) < N_PAIRS:
                pos_texts.append(rec["text"])
            elif rec["label"] == 0 and len(neg_texts) < N_PAIRS:
                neg_texts.append(rec["text"])
            if len(pos_texts) >= N_PAIRS and len(neg_texts) >= N_PAIRS:
                break

    concept_data[concept] = (pos_texts, neg_texts)
    capped = len(pos_texts) < N_PAIRS or len(neg_texts) < N_PAIRS
    flag = f"  ⚠ capped at {len(pos_texts)}" if capped else ""
    print(f"{concept:20s}: {len(pos_texts)} pos, {len(neg_texts)} neg{flag}")

print(f"\n{len(CONCEPTS)} concepts loaded from GitHub.")
print(f"Compare: the SAE tutorial uses 8 unstructured prompts at 1 layer.")
print(f"We use {N_PAIRS*2} structured texts per concept across all {n_layers} layers.")

# 2) Layer-Wise Decomposition: Three Metrics Instead of Sparse Features

The SAE tutorial decomposes activations at one layer into sparse features using a learned dictionary.

The CAZ framework decomposes the **process** of concept formation across layers using three metrics:

**Separation** $S(l)$ — Fisher-normalized centroid distance. How far apart are the model's internal representations for "concept present" vs. "concept absent" at layer $l$? High separation means the model has made the concept geometrically distinguishable.

$$S(l) = \frac{\|\mu_{+}^{(l)} - \mu_{-}^{(l)}\|}{\sqrt{\frac{1}{2}(\sigma_{+}^{(l)2} + \sigma_{-}^{(l)2})}}$$

**Coherence** $C(l)$ — Explained variance of the leading PCA component of the combined activations. Has the concept crystallized into a single clean direction, or is it smeared across many dimensions?

**Velocity** $v(l)$ — Rate of change of separation: $v(l) = S(l) - S(l-1)$ (smoothed). Positive means the concept is being actively constructed at this layer. Negative means it's being degraded or reallocated.

Together they tell a story the SAE approach can't: **where construction begins, where it peaks, and where the geometry gets repurposed.**

## 2.1 Extract activations at every layer

The SAE tutorial caches activations at one hook point (`blocks.8.hook_resid_pre`).

We extract the residual stream at **every** layer boundary using `output_hidden_states=True`. This is architecture-agnostic — it works identically on GPT-2, Llama, Gemma, Pythia, or any HuggingFace model.

In [ ]:
# ============================================================
# 2.1 Extract contrastive activations for all concepts, all layers
# ============================================================
# This is the core data collection step. For each concept, we run all
# positive and negative texts through the model and collect the residual
# stream state at every layer boundary.
#
# Result: concept_acts[concept] = list of (pos_acts, neg_acts) per layer
#   pos_acts shape: [N_PAIRS, d_model]
#   neg_acts shape: [N_PAIRS, d_model]

concept_acts = {}
for concept in CONCEPTS:
    pos_texts, neg_texts = concept_data[concept]
    layer_pairs = extract_contrastive_activations(
        model, tokenizer, pos_texts, neg_texts,
        device=device, batch_size=8, pool="last",
    )
    concept_acts[concept] = layer_pairs
    print(f"{concept:20s}: {len(layer_pairs)} layers, "
          f"pos shape {layer_pairs[0][0].shape}, neg shape {layer_pairs[0][1].shape}")

print(f"\nExtracted activations for {len(CONCEPTS)} concepts x {len(layer_pairs)} layers.")
print(f"Compare: the SAE tutorial extracts 1 layer x 8 prompts.")

## 2.2 Compute CAZ metrics across depth

Now we compute separation, coherence, and velocity at every layer for each concept. This is the CAZ equivalent of the SAE tutorial's "encode → inspect sparse features" step — but instead of a feature dictionary, we get a **profile** of how each concept assembles.

In [ ]:
# ============================================================
# 2.2 Compute layer-wise metrics and detect CAZ regions
# ============================================================
# min_prominence_frac controls how tall a secondary peak must be relative
# to the tallest peak to count as a separate assembly region.
# Default (0.10) is tuned for larger models; use find_caz_regions_scored
# (min_prominence_frac=0.005) to match the paper's scored analysis and
# detect subtle peaks in small models.

from rosetta_tools.caz import find_caz_regions_scored

concept_metrics = {}
concept_profiles = {}

for concept in CONCEPTS:
    metrics = compute_layer_metrics(concept_acts[concept])
    profile = find_caz_regions_scored(metrics)

    concept_metrics[concept] = metrics
    concept_profiles[concept] = profile

    dominant = profile.dominant
    print(f"{concept:20s}: {profile.n_regions} region(s), "
          f"dominant peak at layer {dominant.peak} ({dominant.peak / n_layers * 100:.0f}% depth), "
          f"S={dominant.peak_separation:.3f}, C={dominant.peak_coherence:.3f}")

print(f"\nUsing find_caz_regions_scored (min_prominence_frac=0.005).")
print(f"Matches the threshold used in the CAZ scaling paper.")

## 2.3 Visualize the assembly process

The SAE tutorial's visualization is a bar chart of top-k feature activations at one layer.

Our visualization shows the full separation curve across all layers. Each concept gets its own profile. Look for:
- **Where does separation rise?** (concept construction begins)
- **Where does it peak?** (maximum geometric distinguishability)
- **Is there more than one peak?** (multimodal assembly — the snapshot approach misses this)
- **Where does velocity go negative?** (the geometry is being degraded or reallocated)

In [ ]:
# ============================================================
# 2.3 Visualize: Separation, Coherence, and Velocity across layers
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.patch.set_facecolor('#FAFAFA')

colors = plt.cm.Set2(np.linspace(0, 1, len(CONCEPTS)))

depth_pct = np.array([lm.layer / n_layers * 100 for lm in concept_metrics[CONCEPTS[0]]])

for idx, concept in enumerate(CONCEPTS):
    metrics_list = concept_metrics[concept]
    c = colors[idx]

    sep_arr = np.array([lm.separation for lm in metrics_list])
    coh_arr = np.array([lm.coherence for lm in metrics_list])
    vel_arr = np.array([lm.velocity for lm in metrics_list])

    axes[0].plot(depth_pct, sep_arr, color=c, label=concept, linewidth=1.8)
    axes[1].plot(depth_pct, coh_arr, color=c, linewidth=1.8)
    axes[2].plot(depth_pct, vel_arr, color=c, linewidth=1.8)

    # Mark ALL detected CAZ peaks for this concept
    profile = concept_profiles[concept]
    for region in profile.regions:
        peak_depth = region.peak / n_layers * 100
        is_dominant = region.peak == profile.dominant.peak
        marker_size = 90 if is_dominant else 45
        marker_style = 'o' if is_dominant else 'D'
        axes[0].scatter(peak_depth, sep_arr[region.peak],
                        color=c, s=marker_size, zorder=5,
                        edgecolors='white' if is_dominant else c,
                        linewidths=1.5,
                        marker=marker_style,
                        alpha=1.0 if is_dominant else 0.6)

# SAE hook point reference line
sae_depth = 8 / n_layers * 100
for ax in axes:
    ax.axvline(sae_depth, color='red', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_facecolor('#FAFAFA')
    ax.grid(axis='both', alpha=0.3)

axes[0].set_ylabel('Separation S(l)')
axes[0].set_title('Concept Assembly Profiles — GPT-2 Small', fontsize=13, fontweight='bold')
axes[0].legend(loc='upper left', fontsize=8, ncol=2)
axes[0].text(sae_depth + 1, axes[0].get_ylim()[1] * 0.95, 'SAE hook\n(layer 8)',
             color='red', fontsize=8, alpha=0.7)

axes[1].set_ylabel('Coherence C(l)')
axes[2].set_ylabel('Velocity v(l)')
axes[2].axhline(0, color='gray', linewidth=0.5)
axes[2].set_xlabel('Model Depth (%)')

# Legend for marker types
from matplotlib.lines import Line2D
legend_markers = [
    Line2D([0], [0], marker='o', color='gray', linestyle='None',
           markersize=9, markeredgecolor='white', markeredgewidth=1.5, label='Dominant CAZ peak'),
    Line2D([0], [0], marker='D', color='gray', linestyle='None',
           markersize=6, alpha=0.6, label='Secondary CAZ peak'),
]
axes[0].legend(handles=axes[0].get_lines() + legend_markers,
               loc='upper left', fontsize=7, ncol=2)

plt.tight_layout()
plt.show()

print("Circles = dominant CAZ peak. Diamonds = secondary CAZ peaks.")
print("Red dashed line = SAE tutorial's hook point (layer 8).")

## 2.4 The ordering: which concepts assemble first?

Across 34 models and 8 architecture families, we consistently observe the same ordering tendency: concrete/surface concepts assemble earlier, abstract/evaluative concepts assemble deeper. The SAE tutorial can't see this because it looks at one layer.

Let's check whether GPT-2 Small follows the cross-architecture tendency.

In [ ]:
# ============================================================
# 2.4 Concept ordering — does GPT-2 Small follow the pattern?
# ============================================================

# Cross-architecture reference ordering (from 22-model mean, CAZ scaling paper):
REFERENCE_ORDER = ["credibility", "negation", "causation", "temporal_order",
                   "sentiment", "certainty", "moral_valence"]

# This model's ordering by dominant peak depth
model_ordering = sorted(CONCEPTS,
                        key=lambda c: concept_profiles[c].dominant.peak)

print("Cross-architecture reference (22-model mean):")
for i, c in enumerate(REFERENCE_ORDER):
    print(f"  {i+1}. {c}")

print(f"\nGPT-2 Small ordering (this run):")
for i, c in enumerate(model_ordering):
    peak = concept_profiles[c].dominant.peak
    depth = peak / n_layers * 100
    print(f"  {i+1}. {c:20s} — peak at layer {peak} ({depth:.0f}% depth)")

# Quick Kendall's tau against reference
from scipy.stats import kendalltau
ref_ranks = [REFERENCE_ORDER.index(c) for c in CONCEPTS]
model_ranks = [model_ordering.index(c) for c in CONCEPTS]
tau, p = kendalltau(ref_ranks, model_ranks)
print(f"\nKendall's tau vs reference: {tau:.3f} (p={p:.4f})")
if tau > 0 and p < 0.05:
    print("  -> Significant positive correlation with cross-architecture ordering.")
elif tau > 0:
    print("  -> Positive but not individually significant (common for small models).")
else:
    print("  -> GPT-2 Small is one of the outliers (negative tau).")

# 3) Causal Validation: Ablation at the CAZ Peak

The SAE tutorial demonstrates causality in two ways:
1. **Steering** — add an SAE feature direction to amplify a concept.
2. **Clamping to zero** — remove a feature in the SAE's sparse code and reconstruct.

Both operate at a single layer. The CAZ framework asks a different causal question:

> **If concept assembly peaks at layer $L$, does removing the concept direction at $L$ produce the maximum behavioral disruption?**

This is the **Mid-Stream Ablation Hypothesis** (CAZ Prediction 1). We test it by:
1. Computing the dominant concept direction (centroid difference) at the CAZ peak layer.
2. Orthogonally projecting out that direction at the peak layer during a forward pass.
3. Measuring KL divergence between ablated and baseline next-token distributions.
4. Comparing to ablation at non-peak layers.

If the CAZ peak is causally meaningful, ablation there should hurt more than ablation elsewhere.

In [ ]:
# ============================================================
# 3.1 Pick a concept and compute its dominant direction
# ============================================================
# We'll use causation — it peaks at L8 (~67% depth) in GPT-2 Small,
# well before the final layer, giving a clean mid-model causal leverage story.
# (negation peaks at the very end of this small model with limited data;
# causation shows the pattern more clearly at this scale.)

ABLATION_CONCEPT = "causation"
profile = concept_profiles[ABLATION_CONCEPT]
peak_layer = profile.dominant.peak

print(f"Concept: {ABLATION_CONCEPT}")
print(f"Dominant CAZ peak: layer {peak_layer} ({peak_layer/n_layers*100:.0f}% depth)")
print(f"Peak separation: {profile.dominant.peak_separation:.3f}")
print(f"Peak coherence:  {profile.dominant.peak_coherence:.3f}")

# Compute the concept direction at the peak layer
pos_acts_peak = concept_acts[ABLATION_CONCEPT][peak_layer][0]
neg_acts_peak = concept_acts[ABLATION_CONCEPT][peak_layer][1]
direction = compute_dominant_direction(pos_acts_peak, neg_acts_peak)
print(f"\nConcept direction computed at layer {peak_layer} (shape: {direction.shape})")

## 3.2 Layer sweep: ablation at peak vs. off-peak

The SAE tutorial steers at one layer with different $\alpha$ values.

We ablate at **every** layer and compare the behavioral disruption. If the CAZ peak is real, it should be the point of maximum causal leverage.

> This is the key difference between snapshot and process thinking: the SAE tutorial asks "does this feature at layer 8 matter?" The CAZ framework asks "at which layer does this concept matter most, and is it the same layer where we see peak geometric separation?"

> **Scale note**: This visualization is most compelling with larger models or the full dataset (1300+ pairs per concept). With 48 pairs on GPT-2 Small, separation curves can trend monotonically without a clear mid-model peak — there isn't enough depth or data to make the assembly zone obvious. `causation` has the clearest signal at this scale. For a sharper story, rerun with `N_PAIRS = 500` or switch to GPT-2 XL.

In [ ]:
# ============================================================
# 3.2 Ablation sweep across all layers
# ============================================================

from transformers import AutoModelForCausalLM

lm_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).to(device)
lm_model.eval()

test_prompts = [
    "The bridge collapsed because",
    "Scientists concluded that the increase in temperature caused",
    "The policy failed to achieve its goals because",
    "The experiment failed to replicate the original",
]

baseline_logits = compute_baseline_logits(lm_model, tokenizer, test_prompts, device=device)

transformer_layers = get_transformer_layers(lm_model)
kl_by_layer = []

# Build a lookup: transformer block index → CAZ label(s)
profile = concept_profiles[ABLATION_CONCEPT]
block_to_caz = {}
for i, region in enumerate(profile.regions):
    block = region.peak - 1  # LayerMetrics index includes embedding; blocks are 0-indexed
    block = max(0, min(block, n_layers - 1))
    is_dominant = region.peak == profile.dominant.peak
    label = f"CAZ {i+1} peak" + (" (dominant)" if is_dominant else "")
    block_to_caz[block] = label

for layer_idx in range(n_layers):
    pos_l = concept_acts[ABLATION_CONCEPT][layer_idx + 1][0]
    neg_l = concept_acts[ABLATION_CONCEPT][layer_idx + 1][1]
    dir_l = compute_dominant_direction(pos_l, neg_l)

    kls = []
    with DirectionalAblator(transformer_layers[layer_idx], dir_l, dtype=dtype):
        for i, prompt in enumerate(test_prompts):
            enc = tokenizer(prompt, return_tensors="pt").to(device)
            with torch.no_grad():
                ablated_out = lm_model(**enc)
            ablated_logits = ablated_out.logits[0, -1, :].cpu()
            kls.append(kl_divergence_from_logits(baseline_logits[i], ablated_logits))

    mean_kl = np.mean(kls)
    kl_by_layer.append(mean_kl)

    marker = f"  <-- {block_to_caz[layer_idx]}" if layer_idx in block_to_caz else ""
    print(f"  Layer {layer_idx:2d}: KL = {mean_kl:.4f}{marker}")

best_block = int(np.argmax(kl_by_layer))
print(f"\nMax KL disruption at transformer block {best_block}")
for i, region in enumerate(profile.regions):
    block = max(0, min(region.peak - 1, n_layers - 1))
    dom = " (dominant)" if region.peak == profile.dominant.peak else ""
    print(f"CAZ {i+1}{dom}: peak at transformer block {block} ({region.depth_pct:.0f}% depth)")

In [ ]:
# ============================================================
# 3.3 Visualize: KL divergence vs. separation across layers
# ============================================================
# Overlay the ablation disruption (causal effect) with the separation curve.
# We annotate ALL detected CAZ regions, not just the dominant peak —
# showing each individual assembly event.

fig, ax1 = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#FAFAFA')
ax1.set_facecolor('#FAFAFA')

# Separation curve (skip embedding to align with transformer blocks)
metrics_list = concept_metrics[ABLATION_CONCEPT]
sep = np.array([lm.separation for lm in metrics_list[1:]])  # skip embedding
layers_x = np.arange(n_layers)

ax1.plot(layers_x, sep, color='#4A90D9', linewidth=2, label='Separation S(l)')
ax1.set_ylabel('Separation', color='#4A90D9', fontsize=11)
ax1.tick_params(axis='y', labelcolor='#4A90D9')

# KL divergence on secondary axis
ax2 = ax1.twinx()
ax2.bar(layers_x, kl_by_layer, color='#E67E22', alpha=0.45, label='KL divergence (ablation)')
ax2.set_ylabel('KL Divergence', color='#E67E22', fontsize=11)
ax2.tick_params(axis='y', labelcolor='#E67E22')

# Annotate ALL detected CAZ regions
profile = concept_profiles[ABLATION_CONCEPT]
region_colors = ['#2ECC71', '#9B59B6', '#E74C3C', '#1ABC9C']

for i, region in enumerate(profile.regions):
    block = region.peak - 1   # CAZ layer index includes embedding; blocks are 0-indexed
    block = max(0, min(block, n_layers - 1))
    color = region_colors[i % len(region_colors)]
    is_dominant = region.peak == profile.dominant.peak

    linestyle = '--' if is_dominant else ':'
    linewidth = 1.8 if is_dominant else 1.3
    label = f'CAZ {i+1} (dominant)' if is_dominant else f'CAZ {i+1}'

    ax1.axvline(block, color=color, linestyle=linestyle, alpha=0.8, linewidth=linewidth, label=label)
    ax1.text(block + 0.15, ax1.get_ylim()[1] * (0.92 - i * 0.10),
             f'CAZ {i+1}', color=color, fontsize=8, fontweight='bold')

    # Shade the region span
    ax1.axvspan(region.start - 1, region.end - 1, alpha=0.06, color=color)

# SAE comparison: mark where the companion tutorial ablates
SAE_BLOCK = 8  # TransformerLens blocks[8] == transformer block index 8 (0-indexed)
ax1.axvline(SAE_BLOCK, color='#7F8C8D', linestyle='-', alpha=0.7, linewidth=2.0,
            label='SAE ablation (Layer 8)')
ax1.text(SAE_BLOCK + 0.15, ax1.get_ylim()[1] * 0.72,
         'SAE\nLayer 8', color='#7F8C8D', fontsize=8, fontweight='bold')

ax1.set_xlabel('Transformer Block', fontsize=11)
ax1.set_title(
    f'Separation vs. Ablation Disruption — {ABLATION_CONCEPT} in GPT-2 Small',
    fontsize=13, fontweight='bold'
)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)

ax1.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{profile.n_regions} CAZ region(s) detected for {ABLATION_CONCEPT}:")
for i, r in enumerate(profile.regions):
    dom = " (dominant)" if r.peak == profile.dominant.peak else ""
    print(f"  CAZ {i+1}: layers {r.start}–{r.end}, peak at L{r.peak} "
          f"({r.depth_pct:.0f}% depth), S={r.peak_separation:.3f}, score={r.caz_score:.3f}{dom}")
print()
print(f"SAE comparison: the companion tutorial ablates at block {SAE_BLOCK} (Layer 8).")
print(f"This is {'within' if any(r.start - 1 <= SAE_BLOCK <= r.end - 1 for r in profile.regions) else 'outside'} "
      f"a detected CAZ region.")
print("Note: if the KL bars peak at the shallow CAZ rather than the dominant one,")
print("the model may rely more on early assembly for downstream behavior —")
print("something a single-layer SAE analysis would miss entirely.")

In [ ]:
# ============================================================
# 3.4 Ablation target recommender
# ============================================================
# For each detected CAZ region, score it two ways:
#   Geometric score  — caz_score from the CAZ framework (separation prominence
#                      × coherence × width), measures how organized the concept
#                      is at this region.
#   Causal score     — mean KL disruption across blocks in this region, measures
#                      how much ablating here actually changes model behaviour.
#
# Best ablation target = highest causal score (what you want for interventions).
# The geometric score tells you where the concept is LEGIBLE; the causal score
# tells you where it is UPSTREAM. They are often different layers.

profile  = concept_profiles[ABLATION_CONCEPT]
regions  = profile.regions
n_reg    = len(regions)
region_colors = ['#2ECC71', '#9B59B6', '#E74C3C', '#1ABC9C']

# --- Compute per-region causal score (mean KL over blocks in the region) ---
region_stats = []
for i, r in enumerate(regions):
    # region.start / r.end are in LayerMetrics coords (0 = embedding).
    # Transformer block index = LayerMetrics index - 1.
    block_start = max(0, r.start - 1)
    block_end   = min(n_layers - 1, r.end - 1)
    region_kl   = np.mean(kl_by_layer[block_start : block_end + 1])
    peak_kl     = kl_by_layer[max(0, r.peak - 1)]
    region_stats.append({
        "label":        f"CAZ {i+1}\nL{r.start}–{r.end}\n({r.depth_pct:.0f}% depth)",
        "geo_score":    r.caz_score,
        "causal_score": region_kl,
        "peak_kl":      peak_kl,
        "region":       r,
        "color":        region_colors[i % len(region_colors)],
        "is_dominant":  r.peak == profile.dominant.peak,
    })

# --- Find recommended target ---
best_idx = int(np.argmax([s["causal_score"] for s in region_stats]))

# --- Normalize scores for plotting ---
max_geo    = max(s["geo_score"]    for s in region_stats) or 1.0
max_causal = max(s["causal_score"] for s in region_stats) or 1.0
for s in region_stats:
    s["geo_norm"]    = s["geo_score"]    / max_geo
    s["causal_norm"] = s["causal_score"] / max_causal

# --- Plot ---
fig, ax = plt.subplots(figsize=(max(7, n_reg * 2.5), 5))
fig.patch.set_facecolor('#FAFAFA')
ax.set_facecolor('#FAFAFA')

x      = np.arange(n_reg)
width  = 0.32

bars_geo    = ax.bar(x - width/2, [s["geo_norm"]    for s in region_stats],
                     width, label='Geometric score (normalized)',
                     color=[s["color"] for s in region_stats], alpha=0.55)
bars_causal = ax.bar(x + width/2, [s["causal_norm"] for s in region_stats],
                     width, label='Causal score — mean KL (normalized)',
                     color=[s["color"] for s in region_stats], alpha=0.95,
                     edgecolor='white', linewidth=1.2)

# Highlight the recommended target
ax.bar(x[best_idx] + width/2, region_stats[best_idx]["causal_norm"],
       width, color=region_stats[best_idx]["color"],
       edgecolor='black', linewidth=2.0, zorder=5,
       label=f'→ Recommended ablation target (CAZ {best_idx+1})')

ax.set_xticks(x)
ax.set_xticklabels([s["label"] for s in region_stats], fontsize=9)
ax.set_ylabel('Normalised score', fontsize=11)
ax.set_title(
    f'CAZ Ablation Target Recommender — {ABLATION_CONCEPT.upper()} in GPT-2 Small',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=8, loc='upper right')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.25)

# Annotate dominant (geometric) marker
for i, s in enumerate(region_stats):
    if s["is_dominant"]:
        ax.text(x[i] - width/2, s["geo_norm"] + 0.04, "geo\nleader",
                ha='center', fontsize=7, color='#4A90D9', fontweight='bold')

plt.tight_layout()
plt.show()

# --- Text summary ---
best  = region_stats[best_idx]
r_obj = best["region"]
print(f"Concept: {ABLATION_CONCEPT.upper()}")
print()
print(f"  {'CAZ':5s} {'Depth':>8s} {'Geo score':>12s} {'Causal score (KL)':>20s}  {'Recommended?':>14s}")
print(f"  {'─'*5} {'─'*8} {'─'*12} {'─'*20}  {'─'*14}")
for i, s in enumerate(region_stats):
    rec = "  ← BEST TARGET" if i == best_idx else ""
    dom = " (dominant)" if s["is_dominant"] else ""
    print(f"  CAZ {i+1}  {s['region'].depth_pct:>7.0f}%  {s['geo_score']:>12.4f}  "
          f"{s['causal_score']:>20.4f}{rec}{dom}")

print()
print(f"Recommendation: ablate {ABLATION_CONCEPT.upper()} at CAZ {best_idx+1} "
      f"(block {r_obj.peak - 1}, {r_obj.depth_pct:.0f}% depth)")
print()
if best["is_dominant"]:
    print("The geometrically dominant CAZ is also the most causally upstream.")
    print("Legibility and causal leverage align at this layer.")
else:
    dom_idx = next(i for i, s in enumerate(region_stats) if s["is_dominant"])
    print(f"Note: the geometrically dominant peak is CAZ {dom_idx+1}, but CAZ {best_idx+1} ")
    print(f"produces greater behavioural disruption when ablated.")
    print(f"The concept is most LEGIBLE at CAZ {dom_idx+1}, but most CAUSALLY UPSTREAM at CAZ {best_idx+1}.")
    print("A single-peak approach would have targeted the wrong layer.")

## 3.4 Text generation with ablation active

The SAE tutorial generates text with a feature clamped to zero — the SAE encode→edit→decode path runs at every autoregressive step.

We do the exact same thing with `DirectionalAblator`. Because it hooks the layer output, the ablation stays active across every token generation step inside the context manager.

The CAZ-specific comparison the SAE tutorial *cannot* do:
- **Baseline**: no ablation
- **CAZ peak ablation**: remove the concept direction at the dominant assembly layer
- **Off-peak ablation**: remove the same direction at a non-peak layer

If the CAZ peak is the right place to intervene, the peak ablation should produce a more degraded output than the off-peak one.

In [ ]:
# ============================================================
# 3.4 Text generation with CAZ-peak ablation active
# ============================================================

def generate(prompt, max_new_tokens=60, seed=1):
    torch.manual_seed(seed)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = lm_model.generate(
            **enc, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.8, top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def generate_ablated(prompt, layer_idx, direction, max_new_tokens=60, seed=1):
    """Ablation fires on every autoregressive step via the forward hook."""
    torch.manual_seed(seed)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    with DirectionalAblator(transformer_layers[layer_idx], direction, dtype=dtype):
        with torch.no_grad():
            out = lm_model.generate(
                **enc, max_new_tokens=max_new_tokens,
                do_sample=True, temperature=0.8, top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )
    return tokenizer.decode(out[0], skip_special_tokens=True)


# Build per-CAZ ablation targets from the detected profile
profile = concept_profiles[ABLATION_CONCEPT]
caz_targets = []
for i, region in enumerate(profile.regions):
    block = max(0, min(region.peak - 1, n_layers - 1))
    direction = compute_dominant_direction(
        concept_acts[ABLATION_CONCEPT][region.peak][0],
        concept_acts[ABLATION_CONCEPT][region.peak][1],
    )
    is_dominant = region.peak == profile.dominant.peak
    caz_targets.append({
        "label":       f"CAZ {i+1}" + (" (dominant)" if is_dominant else ""),
        "block":       block,
        "depth_pct":   region.depth_pct,
        "direction":   direction,
        "is_dominant": is_dominant,
    })

GEN_PROMPTS = [
    "The bridge collapsed because",
    "Scientists concluded that the increase in temperature caused",
]

print(f"Concept being ablated: {ABLATION_CONCEPT.upper()}")
print(f"{len(caz_targets)} CAZ region(s) detected — generating one ablated output per CAZ.")
print()
print("Note: ablation fires on EVERY autoregressive step. Early tokens may look")
print("intact because the prompt anchors them, but disruption compounds as")
print("generation continues.")

for prompt in GEN_PROMPTS:
    print()
    print("=" * 72)
    print(f"PROMPT: {prompt!r}")
    print()

    print("[ Baseline — no ablation ]")
    print("  Expect: fluent continuation with causal language maintained throughout.")
    print()
    print(generate(prompt))
    print()

    for t in caz_targets:
        dom_note = " — geometrically dominant" if t["is_dominant"] else " — secondary assembly event"
        print(f"[ {ABLATION_CONCEPT.upper()} ablated at {t['label']}{dom_note} ]")
        print(f"  Block {t['block']} ({t['depth_pct']:.0f}% depth)")
        print("  Expect: the opening causal hook may survive (prompt anchoring), but")
        print("  downstream coherence should degrade — looping, topic drift, or loss")
        print("  of the causal thread. Stronger at the causally dominant CAZ.")
        print()
        print(generate_ablated(prompt, t["block"], t["direction"]))
        print()

print("=" * 72)
print(f"Compare outputs across CAZes: does the causally dominant one (highest KL")
print(f"in the sweep above) produce more disruption than the geometrically dominant one?")

# 4) The Dark Matter: What the Single-Peak Assumption Misses

The SAE tutorial's "dark matter" is about reconstruction error — how much of the activation the SAE can't explain. That's an important question.

The CAZ framework reveals a different kind of dark matter: **multimodal assembly**. When the separation curve has multiple peaks, each peak represents a distinct assembly event at a different depth, often encoding the concept in a different geometric direction.

If you only look at the single strongest peak (the standard "best layer" approach), you miss the others. And ablation testing shows that even subtle secondary peaks — what we call **gentle CAZes** — are often causally active.

### The threshold question

Secondary peaks are real but weak. How weak is too weak to count?

`find_caz_regions` (default `min_prominence_frac=0.10`) requires a secondary peak to be at least 10% as prominent as the tallest peak. That threshold is calibrated for larger models where secondary peaks are more distinct.

`find_caz_regions_scored` (default `min_prominence_frac=0.005`) uses a 0.5% threshold — the same setting used in the CAZ scaling paper across 34 models. It detects subtle peaks that the standard threshold would discard.

This choice is itself a scientific decision. The honest answer is:

- **Standard threshold (10%)**: "GPT-2 Small appears unimodal — one clear assembly event per concept."
- **Paper threshold (0.5%)**: "GPT-2 Small has secondary peaks for credibility, certainty, and moral_valence — but they're subtle."

Neither is wrong. The standard SAE workflow has no equivalent choice to make, because it never looks beyond one layer. CAZ forces you to be explicit about what counts as a "real" assembly event — which is a feature, not a bug.

Let's look at which concepts have multimodal profiles under the paper threshold.

In [ ]:
# ============================================================
# 4.1 Multimodal assembly: how many peaks per concept?
# ============================================================

print("Concept Assembly Profiles — GPT-2 Small")
print("=" * 70)
print(f"{'Concept':20s} {'Regions':>8s} {'Multimodal?':>12s}  Peak layers")
print("-" * 70)

multimodal_count = 0
for concept in CONCEPTS:
    profile = concept_profiles[concept]
    n_reg = profile.n_regions
    is_multi = n_reg > 1
    if is_multi:
        multimodal_count += 1

    dominant_peak = profile.dominant.peak
    peaks = []
    for r in profile.regions:
        label = f"L{r.peak}"
        if r.peak == dominant_peak:
            label = f"[{label}]"  # mark dominant
        peaks.append(label)

    multi_label = "YES" if is_multi else "no"
    print(f"{concept:20s} {n_reg:>8d} {multi_label:>12s}  {', '.join(peaks)}")

print("-" * 70)
print(f"{multimodal_count}/{len(CONCEPTS)} concepts have multimodal assembly profiles.")
print(f"\nBrackets mark the dominant peak. Every other peak is a secondary")
print(f"assembly event that the single-peak approach would discard.")

## 4.2 Direction rotation across peaks

When a concept has multiple assembly peaks, are they encoding the same direction or different ones? If the concept direction rotates across depth, a single-layer SAE will only capture one orientation.

Let's measure cosine similarity between the concept direction at each peak for multimodal concepts.

In [ ]:
# ============================================================
# 4.2 Direction rotation: same concept, different geometry at different depths
# ============================================================

from itertools import combinations

for concept in CONCEPTS:
    profile = concept_profiles[concept]
    if profile.n_regions < 2:
        continue

    print(f"\n{concept} — {profile.n_regions} assembly regions")
    peak_layers = [r.peak for r in profile.regions]

    # Compute concept direction at each peak
    directions = {}
    for peak in peak_layers:
        pos = concept_acts[concept][peak][0]
        neg = concept_acts[concept][peak][1]
        directions[peak] = compute_dominant_direction(pos, neg)

    # Pairwise cosine similarity
    for (l1, d1), (l2, d2) in combinations(directions.items(), 2):
        cos_sim = float(np.dot(d1, d2))
        depth1 = l1 / n_layers * 100
        depth2 = l2 / n_layers * 100
        if abs(cos_sim) > 0.7:
            label = "(similar)"
        elif abs(cos_sim) < 0.3:
            label = "(ROTATED)"
        else:
            label = "(partial rotation)"
        print(f"  L{l1} ({depth1:.0f}%) vs L{l2} ({depth2:.0f}%): cos = {cos_sim:.3f}  {label}")

print("\n---")
print("When cosine similarity is low, the concept is encoded in a DIFFERENT direction")
print("at different depths. An SAE trained on one layer's activations would capture")
print("one orientation and miss the other entirely.")

## 4.3 Connecting to SAE dark matter

The SAE tutorial computes reconstruction error at layer 8 and finds that some fraction of activation energy is unexplained — the "dark matter."

The CAZ framework suggests a complementary explanation for why single-layer dark matter exists:

1. **Concepts are in different stages of assembly at any given layer.** At layer 8, some concepts have already peaked and are being degraded. Others haven't peaked yet. The SAE is trying to capture all of them with one dictionary, but they're in different phases of their lifecycle.

2. **Concept directions rotate across depth.** The same concept may be encoded in different geometric orientations at different layers. A dictionary learned at layer 8 captures the orientation at that depth but misses the earlier/later orientations.

3. **Gentle assembly events are below dictionary resolution.** Subtle concept construction happening at layer 8 may be too weak to appear as a top-k SAE feature, but ablation confirms it's doing real work.

These aren't competing explanations — they're complementary views of the same underlying complexity.

## 4.4 Summary (one-slide version)

### The four-act story of this companion

| Act | Central question | Tool used | What we found |
|-----|-----------------|-----------|---------------|
| **1 · Black box** | Why isn't one layer enough? | Model stats, contrastive data | Concepts assemble across the full depth of the model — there is no single "concept layer" |
| **2 · Decomposition** | How does a concept form? | Separation, coherence, velocity curves | Each concept has a characteristic assembly profile; some have multiple peaks at different depths |
| **3 · Causality** | Is the CAZ peak real? | Directional ablation sweep | Removing the concept direction at the CAZ peak produces maximum behavioral disruption — the geometry tracks real computation |
| **4 · Dark matter** | What does the snapshot miss? | Multi-modal detection, direction rotation | Secondary peaks and rotated directions mean single-layer methods systematically undercount concept complexity |

---

### Key quantities to carry forward

$$
\underbrace{S(l)}_{\text{separation}} \quad \underbrace{C(l)}_{\text{coherence}} \quad \underbrace{v(l)}_{\text{velocity}} \quad \rightarrow \quad \underbrace{\text{CAZ profile}}_{\text{all peaks, all layers}}
$$

| Metric | Formula | What it measures |
|--------|---------|-----------------|
| Separation | $\frac{\|\mu_+ - \mu_-\|}{\sqrt{\frac{1}{2}(\sigma_+^2 + \sigma_-^2)}}$ | How distinguishable are concept-present vs concept-absent? |
| Coherence | Leading PCA explained variance | Has the concept crystallized into one clean direction? |
| Velocity | $\Delta S / \Delta l$ (smoothed) | Is the concept being built right now, or has it already formed? |
| Peak KL | KL(baseline ‖ ablated) at CAZ peak | Does geometric separation predict causal importance? |

---

### SAE vs. CAZ — not a competition

| | SAE approach | CAZ approach |
|---|---|---|
| **Asks** | What features exist at this layer? | How do concepts form across layers? |
| **Data** | Unlabeled text (SAE pre-trained) | Contrastive pairs (concept-specific) |
| **Resolution** | One layer, many features | Many layers, one concept per run |
| **Strength** | Feature discovery, steering | Assembly process, cross-layer dynamics |
| **Blind spot** | Which layer to pick; cross-layer dynamics | Doesn't discover features — needs a concept hypothesis |
| **Dependencies** | TransformerLens + SAELens + pre-trained dictionaries | Raw HuggingFace + numpy/scipy |

The ideal workflow uses both: SAE to discover *what* the model represents, CAZ to track *how* it builds those representations.

### Suggested extensions (beyond the bootcamp)

1. **Run the SAE tutorial's layer 8 features through CAZ.**
   Take the top SAE features from the companion tutorial. Build contrastive pairs that activate/don't activate each feature. Run CAZ to see if those features have assembly profiles that peak at layer 8 — or somewhere else entirely.

2. **Cross-architecture comparison.**
   Repeat this notebook on Pythia-410m or OPT-350m. Do the separation profiles have the same shape? Do concepts peak at the same relative depth?

3. **Gentle CAZ ablation.**
   Find the weakest detected CAZ regions (lowest prominence score). Ablate at those layers. How many are causally active despite being below typical detection thresholds?

4. **SAE feature → CAZ direction alignment.**
   At the SAE's hook layer, compute cosine similarity between the top SAE decoder directions and the CAZ concept directions. Are SAE features capturing the same geometric structure that CAZ detects?

5. **Concept ordering across scale.**
   Run this on GPT-2 Small, Medium, Large, and XL. Does the concept ordering (credibility → negation → sentiment → moral valence) hold as the model scales?

---

#### Pointers to go deeper

| Resource | What it gives you |
|----------|-------------------|
| [CAZ Framework paper](https://github.com/jamesrahenry/Concept_Assembly_Zone) | Full methodology, predictions, and multi-model validation |
| [Rosetta Program](https://github.com/jamesrahenry/Rosetta_Program) | Cross-architecture scaling analysis (34 models, 8 families) |
| [rosetta_tools](https://github.com/jamesrahenry/Rosetta_Tools) | Python library for extraction, CAZ metrics, ablation, and alignment |
| [The Shape of Thought](https://waypoint.henrynet.ca/2026-04-07-the-shape-of-thought/) | Blog-length overview of the CAZ findings |
| [Platonic Representation Hypothesis](https://arxiv.org/abs/2405.07987) | The convergence hypothesis that CAZ concept ordering may relate to |

## Export for cross-model comparison

Run this cell to save key results (separation curves, KL profile, CAZ peaks) to `caz_small_results.json`.

The GPT-2 XL companion notebook (`CAZ_Companion_Tutorial_GPT2_XL.ipynb`) loads this file in its comparison section to overlay the two models side-by-side, with both normalized by depth.


In [ ]:
# ============================================================
# Export: save results for cross-model comparison
# ============================================================
# Optional — run this if you want to compare with CAZ_Companion_Tutorial_GPT2_XL.ipynb.
# The XL notebook will load caz_small_results.json from the same directory.

import os

small_results = {
    "model": MODEL_NAME,
    "n_layers": n_layers,
    "concepts": CONCEPTS,
    "ablation_concept": ABLATION_CONCEPT,
    "separation_by_concept": {
        c: [float(lm.separation) for lm in concept_metrics[c]]
        for c in CONCEPTS
    },
    "kl_by_layer": [float(v) for v in kl_by_layer],
    "caz_peaks_by_concept": {
        c: [
            {
                "peak": int(r.peak),
                "depth_pct": float(r.depth_pct),
                "is_dominant": bool(r.peak == concept_profiles[c].dominant.peak),
                "caz_score": float(r.caz_score),
            }
            for r in concept_profiles[c].regions
        ]
        for c in CONCEPTS
    },
}

save_path = "caz_small_results.json"

class _NumpyEncoder(json.JSONEncoder):
    def default(self, o):
        import numpy as np
        if isinstance(o, np.integer): return int(o)
        if isinstance(o, np.floating): return float(o)
        if isinstance(o, np.bool_): return bool(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return super().default(o)

with open(save_path, "w") as f:
    json.dump(small_results, f, cls=_NumpyEncoder)
print(f"Saved to: {os.path.abspath(save_path)}")
print("Run CAZ_Companion_Tutorial_GPT2_XL.ipynb — cell 5.2 will load this for comparison.")
